In [162]:
import geopandas as gpd
import math

import networkx as nx
import folium
import osmnx as ox

In [163]:
def geocode_location(prompt_text):
    """
    :return: geo obj, text location from user input
    """
    while True:
        location = input(prompt_text)
        try:
            # TODO: Ensure location is within city limits
            geo = ox.geocode(location)
            return geo, location
        except Exception:
            print("Location not found. Please query again.")

# Source: ChatGPT
def haversine(lat1, lon1, lat2, lon2):
    R = 6371000  # meters
    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)

    a = math.sin(dphi/2)**2 + \
        math.cos(phi1) * math.cos(phi2) * math.sin(dlambda/2)**2

    return 2 * R * math.asin(math.sqrt(a))

def loc_prompt():
    """
    :return: start and end geo locations
    """
    geo_start, loc_start = geocode_location("Please enter your starting location: ")
    geo_end, loc_end = geocode_location("Please enter your destination: ")

    print(f"Start Point: {geo_start} \'{loc_start}\'")
    print(f"End Point: {geo_end} \'{loc_end}\'")

    points = [geo_start, geo_end]
    return points

def center(points):
    """
    :param points:
    :return: center point and the correct radius for a plot
    """
    start_lat = points[0][0]
    start_lon = points[0][1]
    end_lat = points[1][0]
    end_lon = points[1][1]

    geo_center = (
        ( float(start_lat)+float(end_lat) ) / 2,
        ( float(start_lon)+float(end_lon) ) / 2
                 )

    # Haversine distance between locations * 1.25 for some bezels for mapping
    bezel_factor = 1.25
    geo_dist = (haversine(start_lat, start_lon, end_lat, end_lon) / 2) * bezel_factor
    # geo_dist = max(geo_dist, 1000)
    geo_dist = round(geo_dist, 2)

    return geo_center, geo_dist, start_lat, start_lon, end_lat, end_lon

def compute_route_metrics(graph, origin_node, destination_node):
    # --- Compute routes ---
    route_crime = nx.shortest_path(
        graph, origin_node, destination_node,
        weight='crime_weight', method='dijkstra'
    )

    route_short = nx.shortest_path(
        graph, origin_node, destination_node,
        weight='length', method='dijkstra'
    )

    # --- Helper function to compute path length ---
    def path_weight(graph, route, weight_attr):
        total = 0
        for u, v in zip(route[:-1], route[1:]):
            edge_data = graph.get_edge_data(u, v)

            # Handle MultiDiGraph (multiple edges)
            if isinstance(edge_data, dict):
                edge_data = min(edge_data.values(), key=lambda x: x.get(weight_attr, float('inf')))

            total += edge_data.get(weight_attr, 0)
        return total

    # --- Compute metrics ---
    crime_route_length_m = path_weight(graph, route_crime, 'length')
    crime_route_weighted = path_weight(graph, route_crime, 'crime_weight')
    crime_route_exposure = path_weight(graph, route_crime, 'crime_count') #TODO: reading 0.0 (line 54)

    short_route_length_m = path_weight(graph, route_short, 'length')
    short_route_weighted = path_weight(graph, route_short, 'crime_weight')
    short_route_exposure = path_weight(graph, route_short, 'crime_count') #TODO: reading 0.0 (line 54)

    # --- Print results cleanly ---
    print("\n" + "="*50)
    print("ROUTE COMPARISON")
    print("="*50)

    print("\n[Shortest Distance Route]")
    print(f"  Total Distance (meters):        {short_route_length_m:,.2f}")
    print(f"  Crime-Weighted Distance:        {short_route_weighted:,.2f}")

    print("\n[Safest (Crime-Weighted) Route]")
    print(f"  Total Distance (meters):        {crime_route_length_m:,.2f}")
    print(f"  Crime-Weighted Distance:        {crime_route_weighted:,.2f}")

    print("\n" + "-"*50)
    print("DIFFERENCE")
    print("-"*50)
    print(f"  Extra Distance for Safety (m):  {crime_route_length_m - short_route_length_m:,.2f}")
    print(f"  Crime Reduction Score:                {short_route_exposure - crime_route_exposure:,.2f}")
    print("="*50 + "\n")

In [178]:
# Functions have built in print statements
geo_center, geo_dist, start_lat, start_lon, end_lat, end_lon = center(loc_prompt())

# TODO: ask use what time it is, block the time by 'SHIFT' and only account for the crimes in that time chunk

graph = ox.graph_from_point(geo_center, dist=geo_dist, network_type='walk')
# runtime dependant of graph size


Start Point: (38.8912809, -77.0227351) 'the national garden, washington dc'
End Point: (38.8881349, -77.0198154) 'air and space museum, washington dc'


In [179]:
origin_node = ox.distance.nearest_nodes(graph, start_lon, start_lat)
destination_node = ox.distance.nearest_nodes(graph, end_lon, end_lat)

Crime density weight


In [180]:
crime_gdf = gpd.read_file("../../../Data/Crime/Crime_Incidents_in_2025.geojson")

In [181]:
graph = ox.project_graph(graph)  # projects to UTM automatically
nodes, edges = ox.graph_to_gdfs(graph, nodes=True, edges=True)

crime_gdf = crime_gdf.to_crs(edges.crs)

In [182]:
route_short = nx.shortest_path(graph, origin_node, destination_node, weight = 'length', method = 'dijkstra')

In [183]:
buffer_dist = 41.5  # HYPERPARAM: buffer distance in meters (41.5-65)
                  # TODO: Incorperate into function to allow user to dial in

edges['geometry_buffer'] = edges.geometry.buffer(buffer_dist)

edges_buffered = edges.set_geometry('geometry_buffer')

# spatial join: crimes within each edge buffer
joined = gpd.sjoin(crime_gdf, edges_buffered, how='left', predicate='within')

In [184]:
crime_counts = joined.groupby(['u', 'v', 'key']).size()

edges['crime_count'] = edges.index.map(crime_counts).fillna(0)
edges['crime_density'] = edges['crime_count'] / edges['length']

In [185]:
alpha = 100  # HYPERPARAM: tune this (0 -> 100+)
            # TODO: Incorperate into function to allow user to dial in


In [186]:
cd_max = edges['crime_density'].max()

edges['crime_density_norm'] = (edges['crime_density'] / cd_max)

edges['crime_weight'] = edges['length'] * (1 + (alpha * edges['crime_density_norm']))

In [187]:
for u, v, k, data in graph.edges(keys=True, data=True):
    if (u, v, k) in edges.index:
        data['crime_weight'] = edges.loc[(u, v, k), 'crime_weight']
    else:
        print('WARING: fallback to length occurred')
        data['crime_weight'] = data['length']  # fallback

In [188]:

graph = ox.graph_from_gdfs(nodes, edges) # rebuilding with updated edges values

route_crime = nx.shortest_path(graph, origin_node, destination_node, weight = 'crime_weight', method = 'dijkstra')

In [189]:
# Claude solution for mis-matched graphical representations
graph_latlon = ox.project_graph(graph, to_crs='epsg:4326')

route_coords_short = [
    (graph_latlon.nodes[node]['y'], graph_latlon.nodes[node]['x'])
    for node in route_short
]

route_coords_crime = [
    (graph_latlon.nodes[node]['y'], graph_latlon.nodes[node]['x'])
    for node in route_crime
]

origin_xy = (graph_latlon.nodes[origin_node]['y'], graph_latlon.nodes[origin_node]['x'])
destination_xy = (graph_latlon.nodes[destination_node]['y'], graph_latlon.nodes[destination_node]['x'])

Folium Graph


In [190]:
from folium.plugins import ScrollZoomToggler

m = folium.Map(
    location = geo_center,
    zoom_start = 12 #TODO: zoom should start dependant on how large the route is
)

ScrollZoomToggler().add_to(m)
folium.PolyLine(route_coords_short, color = 'red', weight = 7, opacity = 0.5).add_to(m)
folium.PolyLine(route_coords_crime, color = 'blue', weight = 3, opacity = 0.5).add_to(m)

folium.Marker(
    location = origin_xy,
    icon = folium.Icon(color='green', icon = 'flag', prefix='fa')
).add_to(m)
folium.Marker(
    location = destination_xy,
    icon = folium.Icon(color='red', icon = 'flag', prefix = 'fa')
).add_to(m)

m

In [191]:
compute_route_metrics(graph, origin_node, destination_node)


ROUTE COMPARISON

[Shortest Distance Route]
  Total Distance (meters):        563.11
  Crime-Weighted Distance:        2,144.45

[Safest (Crime-Weighted) Route]
  Total Distance (meters):        649.73
  Crime-Weighted Distance:        649.73

--------------------------------------------------
DIFFERENCE
--------------------------------------------------
  Extra Distance for Safety (m):  86.62
  Crime Reduction:                18.00

